# Atividade 2.3 — Simulador de Memória ROM

**Docente:** Claudomiro de Souza Sales Junior  
**Discente:** Ana Beatriz Monteiro da Silva  
**Curso:** Inteligência Artificial  

Disciplina: Organização e Arquitetura de Computadores — atividade baseada na **Figura 4.18 (slide 77 de "Memória Principal.pptx.pdf")**: *Memória ROM com 4 células de 4 bits cada*.

## Índice

- [1. Enunciado](#enunciado)
- [2. Princípio de funcionamento da ROM](#a23-principio)
- [3. Decodificador 2 para 4 e o sinal de habilitação](#decodificador)
- [4. Tabela dos endereços e conteúdos](#tabela)
- [5. Código do simulador](#a23-codigo)
- [6. Testes dos quatro endereços e da habilitação](#a23-testes)
- [7. Conclusão](#a23-conclusao)
- [Registro técnico](#a23-registro)

<a id="enunciado"></a>
## 1. Enunciado

Modelar, em um simulador de **somente leitura** escrito em Python, a memória **ROM** apresentada na Figura 4.18 (slide 77 de "Memória Principal.pptx.pdf"). A ROM possui:

- 2 entradas de endereço: **A₀** e **A₁**;
- um **decodificador 2 para 4**;
- um sinal de **habilitação** (*Habilitar*);
- 4 saídas de dados: **S₃, S₂, S₁ e S₀**;
- 4 células (palavras), cada uma com 4 bits, já gravadas:

| Endereço | Conteúdo |
|:---:|:---:|
| 00 | 0101 |
| 01 | 1011 |
| 10 | 1110 |
| 11 | 0000 |

O simulador deve: validar se o endereço possui exatamente 2 bits, mostrar a saída para os quatro endereços, exibir a tabela-verdade, explicar como o decodificador seleciona a palavra e **recusar qualquer tentativa de escrita**, pois uma ROM não permite gravação durante o uso.

<a id="a23-principio"></a>
## 2. Princípio de funcionamento da ROM

**ROM** vem do inglês *Read Only Memory*, ou seja, **memória somente de leitura**. Suas principais características são:

- O conteúdo é **gravado previamente** e, **durante o uso normal, só pode ser lido** — não é possível escrever nele.
- É uma memória **não volátil**: o conteúdo permanece guardado mesmo sem energia.

Isso a diferencia da memória **RAM**, que permite tanto leitura quanto escrita durante o funcionamento.

Na ROM da Figura 4.18, ler um endereço significa **selecionar uma das 4 palavras gravadas** e colocar os seus 4 bits nas saídas **S₃ S₂ S₁ S₀**. Quem faz essa seleção é o decodificador, explicado a seguir.

<a id="decodificador"></a>
## 3. Decodificador 2 para 4 e o sinal de habilitação

**Decodificador 2 para 4:** recebe os **2 bits de endereço** (A₁ e A₀) e ativa **exatamente uma** de suas **4 linhas de saída** (00, 01, 10 e 11). Como são 2 entradas, existem 2² = **4 combinações possíveis**, e cada combinação corresponde a uma das 4 células da ROM.

A linha ativada é escolhida pelo valor do endereço, somando os pesos dos bits:

- A₁ é o bit da esquerda e tem **peso 2**;
- A₀ é o bit da direita e tem **peso 1**.

Por exemplo, o endereço **10** significa A₁ = 1 e A₀ = 0, que seleciona a linha 1×2 + 0×1 = **linha 2** (a palavra 1110). A linha ativada conduz os bits dessa palavra até as saídas S₃ S₂ S₁ S₀ por meio das portas OR mostradas na figura.

**Sinal de habilitação (*Habilitar*):** funciona como uma chave geral da ROM.

- Com **Habilitar = 1**, o decodificador fica ativo e a palavra selecionada aparece nas saídas.
- Com **Habilitar = 0**, nenhuma linha do decodificador é ativada. Como a figura usa portas OR e não mostra buffers tri-state, este modelo considera as saídas em **0000**.

<a id="tabela"></a>
## 4. Tabela dos endereços e conteúdos

A tabela abaixo é a mesma da Figura 4.18. O endereço é lido como **A₁A₀** e o conteúdo como **S₃S₂S₁S₀** (o bit da esquerda é S₃ e o da direita é S₀).

| Endereço (A₁A₀) | Conteúdo (S₃S₂S₁S₀) | S₃ | S₂ | S₁ | S₀ |
|:---:|:---:|:---:|:---:|:---:|:---:|
| 00 | 0101 | 0 | 1 | 0 | 1 |
| 01 | 1011 | 1 | 0 | 1 | 1 |
| 10 | 1110 | 1 | 1 | 1 | 0 |
| 11 | 0000 | 0 | 0 | 0 | 0 |

<a id="a23-codigo"></a>
## 5. Código do simulador

A ideia já foi explicada acima; agora vem o código. Para ficar fácil de entender, a ROM é guardada em **duas listas que andam juntas**: uma com os endereços e outra com os conteúdos. A leitura é feita **passo a passo**, do mesmo jeito que o circuito faria: conferir o endereço, ativar a linha do decodificador, ler a palavra e mostrar as saídas. O endereço a consultar fica guardado em uma **variável**, assim o notebook roda do início ao fim sem travar e sempre mostra o mesmo resultado.

In [1]:
# A memoria ROM do slide 77 tem 4 celulas e cada celula guarda 4 bits.
# Como ainda estou aprendendo, vou guardar a ROM em duas listas simples.
# A lista "enderecos" tem os 4 enderecos possiveis (2 bits cada).
# A lista "conteudos" tem a palavra de 4 bits guardada em cada endereco.
# As duas listas andam juntas: a posicao 0 de uma combina com a posicao 0 da outra.

enderecos = ["00", "01", "10", "11"]
conteudos = ["0101", "1011", "1110", "0000"]

# Esta ROM e SOMENTE LEITURA: o conteudo acima foi gravado antes
# e nao pode ser alterado enquanto a memoria esta sendo usada.

# Vou mostrar a ROM para conferir se ficou igual a tabela do slide 77.
print("Conteudo gravado na ROM (somente leitura):")
for posicao in range(4):
    print("Endereco " + enderecos[posicao] + "  ->  Conteudo " + conteudos[posicao])

Conteudo gravado na ROM (somente leitura):
Endereco 00  ->  Conteudo 0101
Endereco 01  ->  Conteudo 1011
Endereco 10  ->  Conteudo 1110
Endereco 11  ->  Conteudo 0000


### Leitura de um endereço (passo a passo)

In [2]:
# Aqui eu faco a LEITURA de um endereco da ROM, passo a passo,
# do mesmo jeito que o circuito do slide 77 faria.

# 1) O endereco que eu quero ler.
#    Deixo o endereco numa variavel para o notebook rodar inteiro sem travar.
#    Para testar outro endereco, e so trocar o valor abaixo (ex.: "00", "01", "10", "11").
endereco = "10"

# 2) O sinal de habilitacao (Habilitar): 1 = ROM ligada, 0 = ROM desligada.
habilitar = 1

# 3) Antes de ler, confiro se o endereco tem exatamente 2 bits
#    e se cada caractere e 0 ou 1.
endereco_valido = True
if len(endereco) != 2:
    endereco_valido = False
for bit in endereco:
    if bit != "0" and bit != "1":
        endereco_valido = False

# 4) Agora eu uso o resultado da conferencia.
if endereco_valido == False:
    print("Endereco invalido:", endereco)
    print("O endereco precisa ter exatamente 2 bits, usando so 0 e 1 (ex.: 00, 01, 10, 11).")
else:
    # O decodificador 2 para 4 olha os dois bits do endereco.
    # A1 e o bit da esquerda (peso 2) e A0 e o bit da direita (peso 1).
    A1 = endereco[0]
    A0 = endereco[1]

    # Monto as 4 linhas do decodificador comecando todas em 0.
    # So UMA delas vai virar 1 (a linha escolhida pelo endereco).
    linhas = [0, 0, 0, 0]

    # Descubro qual linha ligar somando os pesos dos bits.
    posicao = 0
    if A1 == "1":
        posicao = posicao + 2
    if A0 == "1":
        posicao = posicao + 1

    if habilitar == 1:
        # Com a ROM habilitada, o decodificador liga a linha escolhida.
        linhas[posicao] = 1

        # A linha ligada seleciona a palavra guardada naquele endereco.
        palavra = conteudos[posicao]

        # As saidas S3 S2 S1 S0 sao os 4 bits dessa palavra.
        S3 = palavra[0]
        S2 = palavra[1]
        S1 = palavra[2]
        S0 = palavra[3]

        print("Habilitar = 1 (ROM ligada)")
        print("Endereco lido:", endereco, "( A1 =", A1, "/ A0 =", A0, ")")
        print("Linhas do decodificador (00 01 10 11):", linhas)
        print("Linha ativada: linha", posicao)
        print("Palavra lida:", palavra)
        print("Saidas -> S3 =", S3, " S2 =", S2, " S1 =", S1, " S0 =", S0)
    else:
        # Com a ROM desligada, nenhuma linha e ativada.
        print("Habilitar = 0 (ROM desligada)")
        print("Nenhuma linha do decodificador e ativada.")
        print("Saidas -> S3 = 0  S2 = 0  S1 = 0  S0 = 0")

Habilitar = 1 (ROM ligada)
Endereco lido: 10 ( A1 = 1 / A0 = 0 )
Linhas do decodificador (00 01 10 11): [0, 0, 1, 0]
Linha ativada: linha 2
Palavra lida: 1110
Saidas -> S3 = 1  S2 = 1  S1 = 1  S0 = 0


<a id="a23-testes"></a>
## 6. Testes dos quatro endereços e da habilitação

Abaixo testo a leitura dos **quatro endereços** (montando a tabela-verdade), o efeito do sinal de **habilitação** ligado e desligado, e por fim uma **tentativa de escrita**, que deve ser recusada.

### Tabela-verdade dos quatro endereços (Habilitar = 1)

In [3]:
# Agora eu testo os QUATRO enderecos de uma vez, com a ROM habilitada,
# para montar a tabela-verdade da ROM (endereco -> saidas S3 S2 S1 S0).

habilitar = 1

print("Tabela-verdade da ROM (Habilitar = 1)")
print("Habilitar | A1 A0 | S3 S2 S1 S0")
print("-------------------------------")

for posicao in range(4):
    endereco = enderecos[posicao]
    A1 = endereco[0]
    A0 = endereco[1]

    # palavra guardada nesse endereco e seus 4 bits
    palavra = conteudos[posicao]
    S3 = palavra[0]
    S2 = palavra[1]
    S1 = palavra[2]
    S0 = palavra[3]

    print("    " + str(habilitar) + "     |  " + A1 + "  " + A0 + "  |  " + S3 + "  " + S2 + "  " + S1 + "  " + S0)

Tabela-verdade da ROM (Habilitar = 1)


Habilitar | A1 A0 | S3 S2 S1 S0
-------------------------------
    1     |  0  0  |  0  1  0  1
    1     |  0  1  |  1  0  1  1
    1     |  1  0  |  1  1  1  0
    1     |  1  1  |  0  0  0  0


### Validação do endereço (rejeita endereços inválidos)

In [4]:
# Aqui eu testo a VALIDACAO do endereco usando enderecos errados de proposito.
# A ROM so aceita enderecos com exatamente 2 bits, usando apenas 0 e 1.

# Lista com alguns enderecos para testar (uns certos e uns errados).
testes_de_endereco = ["10", "1", "101", "1A"]

for endereco in testes_de_endereco:
    # Repito a mesma conferencia da leitura: tamanho 2 e so caracteres 0 ou 1.
    endereco_valido = True
    if len(endereco) != 2:
        endereco_valido = False
    for bit in endereco:
        if bit != "0" and bit != "1":
            endereco_valido = False

    if endereco_valido == True:
        print("Endereco", endereco, "-> valido (tem 2 bits 0/1).")
    else:
        print("Endereco", endereco, "-> INVALIDO (precisa ter exatamente 2 bits, so 0 e 1).")

Endereco 10 -> valido (tem 2 bits 0/1).
Endereco 1 -> INVALIDO (precisa ter exatamente 2 bits, so 0 e 1).
Endereco 101 -> INVALIDO (precisa ter exatamente 2 bits, so 0 e 1).
Endereco 1A -> INVALIDO (precisa ter exatamente 2 bits, so 0 e 1).


### Efeito do sinal de habilitação (Habilitar = 1 e Habilitar = 0)

In [5]:
# Aqui eu mostro o efeito da habilitacao usando o MESMO endereco
# em duas situacoes, para comparar.

endereco = "01"

# Descubro a posicao desse endereco (A1 peso 2, A0 peso 1).
posicao = 0
if endereco[0] == "1":
    posicao = posicao + 2
if endereco[1] == "1":
    posicao = posicao + 1

palavra = conteudos[posicao]

print("Endereco testado:", endereco)

# Situacao 1: ROM habilitada
habilitar = 1
if habilitar == 1:
    print("Habilitar = 1 -> saidas S3 S2 S1 S0 =", palavra[0], palavra[1], palavra[2], palavra[3])

# Situacao 2: ROM desabilitada
habilitar = 0
if habilitar == 0:
    print("Habilitar = 0 -> nenhuma linha e ativada; saidas S3 S2 S1 S0 = 0 0 0 0")

Endereco testado: 01
Habilitar = 1 -> saidas S3 S2 S1 S0 = 1 0 1 1
Habilitar = 0 -> nenhuma linha e ativada; saidas S3 S2 S1 S0 = 0 0 0 0


### Tentativa de escrita (deve ser recusada)

In [6]:
# A ROM e somente leitura. Mesmo assim, vou simular alguem TENTANDO gravar
# um valor novo, so para mostrar que a ROM nao deixa escrever durante o uso.

# Exemplo de tentativa: gravar a palavra 1111 no endereco 00.
endereco_para_gravar = "00"
valor_para_gravar = "1111"

# Como a memoria e ROM, eu NAO altero as listas. So aviso que nao da para gravar.
print("Tentativa de escrita:")
print("Pediram para gravar", valor_para_gravar, "no endereco", endereco_para_gravar)
print("ROM significa Read Only Memory (memoria somente de leitura).")
print("Durante o uso a ROM NAO permite gravacao, entao a escrita foi recusada.")

# Confirmo que o conteudo do endereco 00 continua o mesmo de antes.
print("O conteudo do endereco 00 continua sendo:", conteudos[0])

Tentativa de escrita:


Pediram para gravar 1111 no endereco 00
ROM significa Read Only Memory (memoria somente de leitura).
Durante o uso a ROM NAO permite gravacao, entao a escrita foi recusada.
O conteudo do endereco 00 continua sendo: 0101


<a id="a23-conclusao"></a>
## 7. Conclusão

O simulador reproduziu corretamente a memória ROM da Figura 4.18 (slide 77). Para cada um dos quatro endereços, a saída S₃S₂S₁S₀ obtida foi igual ao conteúdo gravado na tabela: 00→0101, 01→1011, 10→1110 e 11→0000. O decodificador 2 para 4 ativou sempre uma única linha, escolhida pelo valor do endereço (A₁ com peso 2 e A₀ com peso 1), e foi essa linha que selecionou a palavra colocada nas saídas.

O sinal de habilitação funcionou como esperado: com *Habilitar* = 1 a palavra apareceu nas saídas e com *Habilitar* = 0 nenhuma linha foi ativada e as saídas ficaram em `0000`. A validação rejeitou endereços que não tivessem exatamente 2 bits 0/1, e a tentativa de escrita foi recusada, confirmando que uma ROM é uma memória **somente de leitura**.

<a id="a23-registro"></a>
## Registro técnico (fontes, hipóteses e testes)

**Fonte obrigatória:** Figura 4.18 — *Memória ROM com 4 células de 4 bits cada* — slide 77 de "Memória Principal.pptx.pdf" (página 77 do arquivo). O slide é apenas visual (sem texto). Os slides 76 e 78 foram consultados somente como contexto e não fazem parte desta atividade.

**Hipóteses assumidas:**
- O endereço é lido como A₁A₀ (bit da esquerda = A₁, peso 2; bit da direita = A₀, peso 1) e o conteúdo como S₃S₂S₁S₀ (bit da esquerda = S₃), conforme os rótulos da figura.
- Com *Habilitar* = 0 nenhuma linha do decodificador é ativada; sem buffers tri-state na figura, o modelo usa saídas `0000`.
- O endereço a consultar fica guardado em uma variável (em vez de `input`), para que o notebook seja reproduzível e execute sem travar, conforme as regras da atividade.

**Testes realizados e resultados:**
- Leitura individual do endereço 10 → saídas 1110 (correto).
- Tabela-verdade dos 4 endereços → 00→0101, 01→1011, 10→1110, 11→0000 (todos corretos).
- Habilitação: endereço 01 com *Habilitar* = 1 → 1011; com *Habilitar* = 0 → saídas 0000 (correto).
- Tentativa de escrita (gravar 1111 no endereço 00) → recusada; conteúdo permaneceu 0101 (correto).
- O notebook foi executado do início ao fim em um kernel reiniciado, sem erros.

**Limitações:**
- É um simulador lógico em software; não substitui o circuito físico nem uma simulação no Logisim.
- O modelo não representa detalhes elétricos; ele reproduz somente o decodificador e as portas OR mostrados no slide.